# WM-811K 취약 클래스 weak-vs-none margin 실험

결함 픽셀을 삭제하지 않고 Scratch, Loc, Edge-Loc의 true logit이 none logit보다 높아지도록 작은 margin penalty를 train에만 적용합니다. Test 추론은 수행하지 않습니다.

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.0
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임에서 T4 GPU를 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

아래 셀은 파일을 하나씩 네 번 선택합니다. 기본 CNN, 기존 취약 클래스 후보, 새 margin 후보 스크립트, split CSV 순서입니다.

In [ ]:
uploads = [
    ('train_wm811k_cnn.py', '1/4: train_wm811k_cnn.py를 선택하세요.'),
    ('train_wm811k_weak_class_candidates.py', '2/4: train_wm811k_weak_class_candidates.py를 선택하세요.'),
    ('train_wm811k_weak_margin_candidates.py', '3/4: train_wm811k_weak_margin_candidates.py를 선택하세요.'),
    ('split_assignments.csv', '4/4: split_assignments.csv를 선택하세요.'),
]
for target_name, message in uploads:
    print(message)
    uploaded = files.upload()
    assert len(uploaded) == 1, '한 번에 파일 하나만 선택하세요.'
    Path('/content', target_name).write_bytes(next(iter(uploaded.values())))
print('업로드 완료')

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
ASSIGNMENTS = Path('/content/split_assignments.csv')
SCRIPT = Path('/content/train_wm811k_weak_margin_candidates.py')
BASELINE_SOURCE = Path('/content/drive/MyDrive/AI_project/WM811K_weak_class_experiment/validation_only_seed42/baseline')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_weak_margin_experiment/validation_only_seed42')
for path in (MAPS, LABELS, ASSIGNMENTS, SCRIPT, Path('/content/train_wm811k_cnn.py'), Path('/content/train_wm811k_weak_class_candidates.py'), BASELINE_SOURCE / 'run_summary.json'):
    assert path.is_file(), f'필수 파일이 없습니다: {path}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(SCRIPT), '--maps', str(MAPS), '--labels', str(LABELS), '--assignments', str(ASSIGNMENTS), '--output-dir', str(OUTPUT_DIR), '--baseline-source', str(BASELINE_SOURCE), '--epochs', '20', '--batch-size', '256', '--num-workers', '2', '--seed', '42']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'학습 스크립트 실패: 종료 코드 {return_code}'
print('Validation-only margin 후보 학습 완료:', OUTPUT_DIR)

In [ ]:
ZIP_BASE = Path('/content/wm811k_weak_margin_validation_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('다운로드 준비:', zip_path, f'{zip_path.stat().st_size / 1024**2:.2f} MB')
files.download(str(zip_path))

다운로드한 `wm811k_weak_margin_validation_results.zip`을 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/`에 압축을 풀지 않고 넣으세요. 기존 baseline은 재사용하므로 새로 학습하는 후보는 2개입니다.